# 03 — RAGAS Evaluation

Evaluates the full RAG chain on QASPER and SciQ using five RAGAS metrics:
- **faithfulness** — grounded in retrieved context
- **answer_relevancy** — semantically relevant to the question
- **context_precision** — useful fraction of retrieved context
- **context_recall** — ground-truth information present in context
- **noise_sensitivity** — stability under irrelevant context

In [ ]:
import subprocess, json, pathlib, glob
import pandas as pd
import matplotlib.pyplot as plt

DATASET = "qasper"  # change to "sciq" for SciQ
N_SAMPLES = 50      # increase for production eval

result = subprocess.run(
    ["python", "-m", "production_rag.evaluation.ragas_suite",
     "--dataset", DATASET, "--n", str(N_SAMPLES)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-3000:])

In [ ]:
# Load latest results file
files = sorted(glob.glob("../eval_results/ragas_*.json"), reverse=True)
latest = json.loads(pathlib.Path(files[0]).read_text())
print(f"Results file: {files[0]}")

scores = latest["scores"]
thresholds = latest["thresholds"]

df = pd.DataFrame({
    "Score": pd.Series(scores),
    "Threshold": pd.Series(thresholds),
})
df["Pass"] = df["Score"] >= df["Threshold"]
df.style.applymap(lambda v: "color: green" if v else "color: red", subset=["Pass"])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(df))
bars = ax.bar(x, df["Score"], color=["#2ecc71" if p else "#e74c3c" for p in df["Pass"]], alpha=0.85)
ax.plot(x, df["Threshold"], "k--", linewidth=1.5, label="CI gate threshold")
ax.set_xticks(x)
ax.set_xticklabels(df.index, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title(f"RAGAS Scores — {DATASET} (n={N_SAMPLES})")
ax.legend()
plt.tight_layout()
plt.savefig("../eval_results/ragas_scores.png", dpi=150)
plt.show()